<a href="https://colab.research.google.com/github/OlaniyiSegunIsrael/Assignment-10-SO/blob/main/Assignment%2010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Assignment 10: Customer Review Sentiment Analysis

- Kaggle Data link- https://www.kaggle.com/datasets/kanchana1990/uber-customer-reviews-dataset-2024
- Git Hub: https://github.com/OlaniyiSegunIsrael/Assignment-10-SO
- Google Colab: https://colab.research.google.com/drive/1fPxxtJHmqUtBrOglyKydyYSormcuWSEv?usp=sharing

In [1]:
!apt-get install -y git
!git config --global user.email "segun.olaniyi@students.williscollege.com"
!git config --global user.name "OlaniyiSegunIsrael"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [2]:
!git clone https://github.com/OlaniyiSegunIsrael/Assignment-10-SO.git
%cd Assignment-10-SO
!ls

Cloning into 'Assignment-10-SO'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 257.71 KiB | 3.35 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Assignment-10-SO
'Assignment 10.ipynb'   README.md


In [3]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [4]:
#Import libraries and set-up
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# NLP embeddings
import spacy

# BERT
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from datasets import Dataset

In [ ]:
#Load Dataset

df = pd.read_csv('../uber_reviews_without_reviewid.csv')

print(df.head())
print(df.info())

In [ ]:
# Check columns
print(df.columns)

# Rename properly (adjust if needed)
df.rename(columns={'content': 'text', 'score': 'label'}, inplace=True)

# Keep only needed columns
df = df[['text', 'label']]

# Drop missing ONLY in required columns
df = df.dropna(subset=['text', 'label'])

# Convert sentiment
def convert_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['label'].apply(convert_sentiment)

# Encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])

# Check before split
print("Final dataset size:", df.shape)

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['sentiment_encoded'], test_size=0.2, random_state=42
)

In [ ]:
#Feature Engineering
bow = CountVectorizer(stop_words='english')
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
#Word Embeddings (SpaCy)
nlp = spacy.load("en_core_web_sm")

def get_embedding(text):
    return nlp(text).vector

X_embeddings = np.array([get_embedding(text) for text in df['text']])

In [ ]:
#Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_embeddings)

plt.scatter(X_pca[:,0], X_pca[:,1], c=df['sentiment_encoded'])
plt.title("PCA of Word Embeddings")
plt.show()

print()

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_embeddings[:1000])

plt.scatter(X_tsne[:,0], X_tsne[:,1], c=df['sentiment_encoded'][:1000])
plt.title("t-SNE Visualization")
plt.show()

In [ ]:
#Traditional Models
#Logistic Regression + Grid Search

lr = LogisticRegression(max_iter=1000)

param_grid_lr = {'C': [0.1, 1, 10]}

grid_lr = GridSearchCV(lr, param_grid_lr, cv=5)
grid_lr.fit(X_train_tfidf, y_train)

best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test_tfidf)

In [ ]:
#Support Vector Machine (SVM)
svm = SVC(probability=True)

param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

grid_svm = GridSearchCV(svm, param_grid_svm, cv=5)
grid_svm.fit(X_train_tfidf, y_train)

best_svm = grid_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test_tfidf)

In [ ]:
#Model Evaluation
#Classification Report
print("Logistic Regression:\n", classification_report(y_test, y_pred_lr))
print("SVM:\n", classification_report(y_test, y_pred_svm))

In [ ]:
#Confusion Matrix
def plot_conf_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

plot_conf_matrix(y_test, y_pred_lr, "LR Confusion Matrix")
plot_conf_matrix(y_test, y_pred_svm, "SVM Confusion Matrix")

In [ ]:
#ROC Curve
y_prob_lr = best_lr.predict_proba(X_test_tfidf)

fpr, tpr, _ = roc_curve(y_test, y_prob_lr[:,1], pos_label=1)
plt.plot(fpr, tpr)
plt.title("ROC Curve (LR)")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.show()

print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr, multi_class='ovr'))

In [ ]:
#BERT Sentiment Analysis
#Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(example):
    return tokenizer(example['text'], padding="max_length", truncation=True)

dataset = Dataset.from_pandas(df[['text', 'sentiment_encoded']])
dataset = dataset.rename_column("sentiment_encoded", "label")
dataset = dataset.map(tokenize_function, batched=True)

train_test = dataset.train_test_split(test_size=0.2)

In [ ]:
#Load Model
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=3)

In [ ]:
#Training
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_test['train'],
    eval_dataset=train_test['test'])

trainer.train()

In [ ]:
#Evaluation
predictions = trainer.predict(train_test['test'])
y_pred_bert = np.argmax(predictions.predictions, axis=1)

print(classification_report(train_test['test']['label'], y_pred_bert))

In [ ]:
#Feature Importance
feature_names = tfidf.get_feature_names_out()
coefficients = best_lr.coef_[0]

top_features = sorted(zip(feature_names, coefficients), key=lambda x: x[1], reverse=True)[:20]

words = [x[0] for x in top_features]
scores = [x[1] for x in top_features]

plt.barh(words, scores)
plt.title("Top Important Words")
plt.show()

In [ ]:
#Model Saving (Deployment)
import joblib

joblib.dump(best_lr, "logistic_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")